# Exercise 10 : Time Series

Run the first cell : it imports pandas and sets `DATA` to the `datasets` folder. The steps use two
files :

- `weather_data_austin_2010.csv` : hourly temperature, dew point and pressure readings for Austin
  in 2010. The file states neither its source nor its time zone
- `austin_airport_departure_data_2015_july.csv` : the scheduled Southwest Airlines departures from
  Austin-Bergstrom airport in July 2015, from the Bureau of Transportation Statistics, in local time

Assume that the departure file lists every scheduled Southwest departure from Austin in July 2015.

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets'

## Step 1 : Parsing

Read `weather_data_austin_2010.csv` into `weather`. Convert the `Date` column with
`pd.to_datetime()` and `format='%Y%m%d %H:%M'` and make it the index with `set_index()`. Print
`weather.head()` and the dtype of the index.

Assign `weather.index.is_monotonic_increasing` to `is_sorted` and the number of duplicated
timestamps to `n_duplicates`, and print both. Why does the order matter before you slice by date?

In [ ]:
weather = pd.read_csv(DATA / 'weather_data_austin_2010.csv')
weather['Date'] = pd.to_datetime(weather['Date'], format='%Y%m%d %H:%M')
weather = weather.set_index('Date')
print(weather.head())
print(weather.index.dtype)

is_sorted = weather.index.is_monotonic_increasing
n_duplicates = weather.index.duplicated().sum()
print(is_sorted, n_duplicates)

## Step 2 : Gaps

A year has 8760 hours. Create the complete hourly range from the first to the last timestamp of
`weather` with `pd.date_range()` and `freq='h'` as `expected`, and print the lengths of `expected`
and `weather`.

Select the timestamps of `expected` without a reading with `expected.difference()` as `missing` and
print them. On which date does the gap fall, and what happens to the clocks in Texas on that date?

In [ ]:
expected = pd.date_range(weather.index.min(), weather.index.max(), freq='h')
print(len(expected), len(weather))
missing = expected.difference(weather.index)
print(missing)

## Step 3 : Filling

Assign `weather['Temperature'].asfreq('h')` to `hourly` and its number of missing values to
`n_filled`.

Fill the gap twice : with `.ffill()` as `filled_ffill`, and with `.interpolate(method='time')` as
`filled_interp`. Print the recorded, forward-filled and interpolated values from 2010-03-14 01:00 to
05:00 side by side. Which estimate suits a temperature better? Would you fill a gap of a week in
the same way?

In [ ]:
hourly = weather['Temperature'].asfreq('h')
n_filled = hourly.isna().sum()
print(n_filled)

filled_ffill = hourly.ffill()
filled_interp = hourly.interpolate(method='time')
filled = pd.DataFrame({'recorded': hourly, 'ffill': filled_ffill, 'interpolated': filled_interp})
print(filled.loc['2010-03-14 01:00':'2010-03-14 05:00'])

## Step 4 : Downsampling

Resample `weather['Temperature']` to days and aggregate with
`.agg(['min', 'max', 'mean', 'count'])` as `daily`. Select the days with fewer than 24 readings as
`incomplete_days` and print them.

Resample the temperatures to month ends with the alias `'ME'` and `.mean()` as `monthly_mean`, and
print it rounded to one decimal. Which month is the warmest? pandas 3 no longer accepts the old
alias `'M'`.

In [ ]:
temperature = weather['Temperature']
daily = temperature.resample('D').agg(['min', 'max', 'mean', 'count'])
print(daily.head())

incomplete_days = daily[daily['count'] < 24]
print(incomplete_days)

monthly_mean = temperature.resample('ME').mean()
print(monthly_mean.round(1))
print(monthly_mean.idxmax().strftime('%B'))

## Step 5 : Rolling mean

Select the August 2010 temperatures with `weather.loc['2010-08', 'Temperature']` as `august`, and
resample them to daily maxima as `daily_highs`.

Smooth the daily highs with `daily_highs.rolling(window=7).mean()` as `smoothed` and print both
Series side by side. Why are the first six values of `smoothed` missing?

In [ ]:
august = weather.loc['2010-08', 'Temperature']
daily_highs = august.resample('D').max()
smoothed = daily_highs.rolling(window=7).mean()
print(pd.DataFrame({'daily_high': daily_highs, 'rolling_7d': smoothed}).head(10))
print(smoothed.isna().sum())

## Step 6 : Event counts

Read `austin_airport_departure_data_2015_july.csv` into `departures` with `skiprows=15`. Strip the
spaces around the column names with `.str.strip()` and drop the footer row, which has no
`'Flight Number'`.

Join `'Date (MM/DD/YYYY)'`, a space and `'Scheduled Departure Time'` into one string per row. Parse
the strings with `format='%m/%d/%Y %H:%M'` into the Series `scheduled` and make it the sorted index
of `departures`.

Four flights have an `'Actual Departure Time'` and a `'Wheels-off Time'` of `'00:00'` : they never
departed, so their delay of 0 was never measured. Mark them in the Boolean Series `not_departed`.
Assign the `'Departure Delay(Minutes)'` column to `delay`, with these four rows set to NaN by
`.where()`.

Count the scheduled departures per day with `departures.resample('D').size()` as
`daily_departures` and print its summary statistics.

In [ ]:
departures = pd.read_csv(DATA / 'austin_airport_departure_data_2015_july.csv', skiprows=15)
departures.columns = departures.columns.str.strip()
departures = departures.dropna(subset=['Flight Number'])

stamps = departures['Date (MM/DD/YYYY)'] + ' ' + departures['Scheduled Departure Time']
scheduled = pd.to_datetime(stamps, format='%m/%d/%Y %H:%M')
departures = departures.set_index(scheduled).sort_index()
print(len(departures), departures.index.min(), departures.index.max())

not_departed = ((departures['Actual Departure Time'] == '00:00')
                & (departures['Wheels-off Time'] == '00:00'))
delay = departures['Departure Delay(Minutes)'].where(~not_departed)
print(not_departed.sum())

daily_departures = departures.resample('D').size()
print(daily_departures.describe())

## Step 7 : Hourly counts

Build the DataFrame `july4` for 4 July 2015 with 24 rows, 00:00 to 23:00, and two columns :
`'departures'`, the number of scheduled departures, and `'mean_delay'`, the mean measured delay per
scheduled hour. Complete the outline in the notebook : reindex onto
`pd.date_range('2015-07-04', periods=24, freq='h')` and fill only the departure counts with 0.
Print `july4`. How many hours have no scheduled departure? Why would a mean delay of 0 be wrong for
those hours?

In [ ]:
hours = pd.date_range('2015-07-04', periods=24, freq='h')
july4 = pd.DataFrame({
    'departures': departures.loc['2015-07-04'].resample('h').size(),
    'mean_delay': delay.loc['2015-07-04'].resample('h').mean(),
}).reindex(hours)
july4['departures'] = july4['departures'].fillna(0).astype('int64')
print(july4)

## Step 8 : Time zones

Sort `scheduled` from Step 6 with `.sort_values()`. Localize it to `'America/Chicago'` as
`scheduled_local`, convert that to `'UTC'` as `scheduled_utc`, and print the first rows of both.

In [ ]:
scheduled = scheduled.sort_values()
scheduled_local = scheduled.dt.tz_localize('America/Chicago')
scheduled_utc = scheduled_local.dt.tz_convert('UTC')
print(pd.DataFrame({'local': scheduled_local, 'utc': scheduled_utc}).head())

## If time permits :

Try `weather.index.tz_localize('America/Chicago')` inside `try`/`except ValueError` and print the
error. Localize again with `nonexistent='NaT'` and `ambiguous='NaT'` as `weather_local`, and select
the timestamps of `weather` that became NaT as `unplaced`. Compare them with the gap from Step 2.
Explain in the answer cell which clock the file seems to use.

Without a solution :

- Plot `daily['min']` and `daily['max']` on one Axes, with axis labels, units and a legend, and save
  the figure with `fig.savefig()`
- Resample `delay` to weeks ending on Sunday with `'W'` and print the mean delay beside the number
  of measured departures per week
- Write two sentences about the departure delays on 4 July, with the number of flights behind each
  figure

In [ ]:
try:
    weather.index.tz_localize('America/Chicago')
except ValueError as error:
    print('ValueError:', error)

weather_local = weather.index.tz_localize('America/Chicago', nonexistent='NaT', ambiguous='NaT')
unplaced = weather.index[weather_local.isna()]
print(unplaced)

Central time skips 02:00 on 14 March 2010 and repeats 01:00 on 7 November. The file has a
reading at 02:00 on 14 March but none at 03:00, and only one reading at 01:00 on 7 November. Its
clock skips an hour in spring, one hour later than Central time does, and never repeats one : it
follows neither Central time nor UTC exactly, so its time zone is unknown.